In [ ]:
from pathlib import Path
import os, json, hashlib, platform, urllib.request, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value):
        print(value)
    def Markdown(value):
        return value

SEED = 20261011
MIN_SWINGS = 5
N_BOOT = 3000
N_BALANCED = 500
DOWNLOAD_MISSING = True

DATA_DIR_OVERRIDE = None

PROJECT_ROOT_OVERRIDE = None

def find_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    cwd = Path.cwd().resolve()
    return next((p for p in [cwd, *cwd.parents]
                 if (p / 'src' / 'obp_utils.py').exists()), cwd)

ROOT = find_root()
RAW = (Path(DATA_DIR_OVERRIDE).expanduser().resolve() if DATA_DIR_OVERRIDE
       else ROOT / 'data' / 'raw' / 'hitting')
RESULTS = ROOT / 'results' / 'swing_plane_consistency'
FIGURES = ROOT / 'figures' / 'swing_plane_consistency'
for folder in (RAW, RESULTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':180,
                    'axes.spines.top':False, 'axes.spines.right':False,
                    'axes.titlesize':13, 'axes.labelsize':11})
BASE_URL = ('https://raw.githubusercontent.com/drivelineresearch/'
            'openbiomechanics/dataset-v1/baseball_hitting/data/')
URLS = {'metadata.csv':BASE_URL + 'metadata.csv',
        'poi_metrics.csv':BASE_URL + 'poi/poi_metrics.csv'}
input_records = []
for filename, url in URLS.items():
    path = RAW / filename
    downloaded = False
    if not path.exists():
        if not DOWNLOAD_MISSING:
            raise FileNotFoundError(f'Place {filename} in {RAW}, or enable downloads.')
        temp = path.with_suffix('.download')
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'OBP-repeatability-study/1.0'})
            with urllib.request.urlopen(req, timeout=60) as response:
                temp.write_bytes(response.read())
            pd.read_csv(temp, nrows=2)
            os.replace(temp, path)
            downloaded = True
        except Exception as exc:
            temp.unlink(missing_ok=True)
            raise RuntimeError(f'Download failed. Download {url} manually into {path}.') from exc
    input_records.append({'file':filename, 'path':str(path), 'source_url':url,
                          'downloaded_this_run':downloaded,
                          'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
print('Project root:', ROOT)
print('Inputs:', RAW)
print('Results:', RESULTS)


In [ ]:
meta = pd.read_csv(RAW / 'metadata.csv', dtype={'user':str, 'session':str, 'session_swing':str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing':str})
meta_cols = ['session_swing','user','session','highest_playing_level',
             'hitter_side','athlete_age','session_height_in','bat_length_in','bat_weight_oz']
poi_cols = ['session_swing','attack_angle_contact_x','bat_speed_mph_contact_x','exit_velo_mph_x']
for name, frame, required in [('metadata',meta,meta_cols),('POI',poi,poi_cols)]:
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')
    if frame['session_swing'].isna().any() or frame['session_swing'].duplicated().any():
        raise ValueError(f'{name} has missing/duplicate session_swing IDs.')
merged = meta[meta_cols].merge(poi[poi_cols], on='session_swing', how='outer',
                              validate='one_to_one', indicator=True)
merged = merged.rename(columns={'user':'hitter','attack_angle_contact_x':'attack_angle_deg',
                                'bat_speed_mph_contact_x':'bat_speed_mph',
                                'exit_velo_mph_x':'exit_velocity_mph'})
for col in ['attack_angle_deg','bat_speed_mph','exit_velocity_mph','athlete_age']:
    merged[col] = pd.to_numeric(merged[col], errors='coerce')
angle = merged['attack_angle_deg']
merged['exclusion_reason'] = np.select(
    [merged['_merge'].ne('both'), merged[['hitter','session']].isna().any(axis=1),
     ~np.isfinite(angle), angle.abs().gt(90)],
    ['unmatched_trial','missing_identity','missing_or_nonfinite_angle','angle_outside_vertical_range'],
    default='retained')
for col in ['bat_speed_mph','exit_velocity_mph']:
    merged.loc[~np.isfinite(merged[col]) | merged[col].le(0), col] = np.nan
trials = merged.loc[merged['exclusion_reason'].eq('retained')].copy()
trials['highest_playing_level'] = trials['highest_playing_level'].fillna('unknown')
for col in ['highest_playing_level','hitter_side']:
    if trials.groupby(['hitter','session'])[col].nunique(dropna=False).gt(1).any():
        raise ValueError(f'Inconsistent {col} within a hitter-session.')
if trials.groupby('hitter')['highest_playing_level'].nunique().gt(1).any():
    raise ValueError('Hitter level changes across sessions; define a longitudinal analysis first.')
if trials.empty:
    raise ValueError('No valid attack angles remain.')
merged.to_csv(RESULTS / 'trial_quality_audit.csv', index=False)
audit = merged['exclusion_reason'].value_counts().rename_axis('status').to_frame('trials')
display(audit)
coverage = trials.groupby('highest_playing_level').agg(
    swings=('session_swing','size'), hitters=('hitter','nunique'), sessions=('session','nunique'))
display(coverage)
print('Attack angle range:', trials.attack_angle_deg.min(), 'to', trials.attack_angle_deg.max(), 'degrees')


In [ ]:
def mad(x):
    x = np.asarray(x, dtype=float)
    return float(np.median(np.abs(x - np.median(x))))

def summarize_sessions(frame, minimum=5, outcome=None):
    needed = ['attack_angle_deg'] + ([outcome] if outcome else [])
    f = frame.dropna(subset=needed)
    records = []
    for (hitter, session), g in f.groupby(['hitter','session'], sort=True):
        if len(g) < minimum:
            continue
        x = g.attack_angle_deg.to_numpy()
        row = {'hitter':hitter, 'session':session,
               'highest_playing_level':g.highest_playing_level.iloc[0],
               'hitter_side':g.hitter_side.iloc[0], 'n_swings':len(g),
               'angle_mean_deg':x.mean(), 'angle_median_deg':np.median(x),
               'angle_sd_deg':x.std(ddof=1), 'angle_mad_deg':mad(x),
               'angle_iqr_deg':np.quantile(x,.75)-np.quantile(x,.25)}
        if outcome:
            row['outcome_mean'] = g[outcome].mean()
            row['outcome_sd'] = g[outcome].std(ddof=1)
        records.append(row)
    return pd.DataFrame(records)

def summarize_hitters(sessions):
    if sessions.empty:
        return pd.DataFrame()
    metrics = ['angle_mean_deg','angle_median_deg','angle_sd_deg','angle_mad_deg','angle_iqr_deg']
    metrics += [c for c in ['outcome_mean','outcome_sd'] if c in sessions]
    agg = {c:'mean' for c in metrics}
    agg.update({'highest_playing_level':'first','n_swings':'sum','session':'nunique'})
    return sessions.groupby('hitter', as_index=False).agg(agg).rename(columns={'session':'n_sessions'})

sessions = summarize_sessions(trials, MIN_SWINGS)
if sessions.empty:
    raise ValueError(f'No hitter-sessions have at least {MIN_SWINGS} valid swings.')
hitters = summarize_hitters(sessions)
sessions.to_csv(RESULTS / 'session_repeatability.csv', index=False)
hitters.to_csv(RESULTS / 'hitter_repeatability.csv', index=False)
display(hitters.sort_values('angle_sd_deg').head(12))
print(f'{len(sessions)} qualifying sessions, {len(hitters)} hitters, {sessions.n_swings.sum()} swings.')
print('The sorted display is descriptive; short sessions do not support a definitive hitter ranking.')


In [ ]:
LEVELS = ['high_school','college','independent','milb']
COLORS = {'high_school':'#9b8ec4','college':'#3973ac','independent':'#d99842','milb':'#318c74'}

def bootstrap_mean(x, seed, minimum=5):
    x = np.asarray(x, float)
    if len(x) < minimum:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = x[rng.integers(0,len(x),size=(N_BOOT,len(x)))].mean(axis=1)
    return tuple(np.quantile(draws,[.025,.975]))

level_rows = []
for i, (level,g) in enumerate(hitters.groupby('highest_playing_level', sort=True)):
    lo, hi = bootstrap_mean(g.angle_sd_deg, SEED+i)
    level_rows.append({'level':level, 'n_hitters':len(g), 'n_sessions':g.n_sessions.sum(),
                      'n_swings':g.n_swings.sum(), 'mean_hitter_sd_deg':g.angle_sd_deg.mean(),
                      'median_hitter_sd_deg':g.angle_sd_deg.median(),
                      'mean_hitter_mad_deg':g.angle_mad_deg.mean(),
                      'ci_low':lo,'ci_high':hi,'sparse_group':len(g)<10})
level_summary = pd.DataFrame(level_rows)

def group_contrast(table, metric, seed):
    a = table.loc[table.highest_playing_level.eq('milb'),metric].dropna().to_numpy()
    b = table.loc[table.highest_playing_level.eq('college'),metric].dropna().to_numpy()
    row = {'metric':metric, 'contrast':'MiLB minus college', 'n_milb':len(a), 'n_college':len(b),
           'difference_deg':a.mean()-b.mean() if len(a) and len(b) else np.nan,
           'ci_low':np.nan,'ci_high':np.nan}
    if min(len(a),len(b)) >= 5:
        rng = np.random.default_rng(seed)
        d = (a[rng.integers(0,len(a),(N_BOOT,len(a)))].mean(axis=1)
             - b[rng.integers(0,len(b),(N_BOOT,len(b)))].mean(axis=1))
        row['ci_low'], row['ci_high'] = np.quantile(d,[.025,.975])
    return row

contrasts = pd.DataFrame([group_contrast(hitters, m, SEED+100+i)
                         for i,m in enumerate(['angle_sd_deg','angle_mad_deg'])])
display(level_summary.round(3))
display(contrasts.round(3))
level_summary.to_csv(RESULTS / 'playing_level_summary.csv', index=False)
contrasts.to_csv(RESULTS / 'milb_college_contrasts.csv', index=False)


In [ ]:
def spearman(x,y):
    x,y = np.asarray(x,float),np.asarray(y,float)
    if len(x)<3 or np.ptp(x)==0 or np.ptp(y)==0:
        return np.nan,np.nan
    result = stats.spearmanr(x,y)
    return float(result.statistic),float(result.pvalue)

def correlation_bootstrap(table, ycol, seed):
    t = table.dropna(subset=['angle_sd_deg',ycol])
    x,y = t.angle_sd_deg.to_numpy(),t[ycol].to_numpy()
    row = {'n_hitters':len(t),'rho':np.nan,'p_value':np.nan,'ci_low':np.nan,'ci_high':np.nan}
    if len(t)<10:
        return row
    row['rho'],row['p_value'] = spearman(x,y)
    if not np.isfinite(row['rho']):
        return row
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(N_BOOT):
        ix = rng.integers(0,len(t),len(t))
        r,_ = spearman(x[ix],y[ix])
        if np.isfinite(r):
            vals.append(r)
    if len(vals)>=N_BOOT*.9:
        row['ci_low'],row['ci_high'] = np.quantile(vals,[.025,.975])
    return row

def bh_adjust(values):
    values = np.asarray(values,float)
    result = np.full(len(values),np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return result
    order = valid[np.argsort(values[valid])]

    q = values[order]*len(values)/np.arange(1,len(order)+1)
    result[order] = np.minimum(1,np.minimum.accumulate(q[::-1])[::-1])
    return result

outcome_tables = {}
association_rows = []
specs = [('mean_exit_velocity','exit_velocity_mph','outcome_mean','Mean exit velocity (mph)'),
         ('exit_velocity_variability','exit_velocity_mph','outcome_sd','Exit-velocity SD (mph)'),
         ('mean_bat_speed','bat_speed_mph','outcome_mean','Mean bat speed (mph)')]
for i,(label,outcome,ycol,ylabel) in enumerate(specs):
    matched_sessions = summarize_sessions(trials,MIN_SWINGS,outcome)
    ht = summarize_hitters(matched_sessions)
    outcome_tables[label] = ht
    ht.to_csv(RESULTS / f'{label}_matched_hitters.csv',index=False)
    row = ({'n_hitters':0,'rho':np.nan,'p_value':np.nan,'ci_low':np.nan,'ci_high':np.nan}
           if ht.empty else correlation_bootstrap(ht,ycol,SEED+200+i))
    row.update({'relationship':label,'minimum_swings':MIN_SWINGS})
    association_rows.append(row)
associations = pd.DataFrame(association_rows)
associations['fdr_q_value'] = bh_adjust(associations.p_value)
display(associations.round(3))
associations.to_csv(RESULTS / 'exploratory_associations.csv',index=False)


In [ ]:
sensitivity_rows = []
for minimum in [4,5,6]:
    ss = summarize_sessions(trials,minimum)
    hh = summarize_hitters(ss)
    if hh.empty:
        sensitivity_rows.append({'minimum_swings':minimum,'n_hitters':0})
        continue
    row = group_contrast(hh,'angle_sd_deg',SEED+300+minimum)
    row.update({'minimum_swings':minimum,'n_hitters':len(hh), 'n_sessions':len(ss),
                'n_swings':int(ss.n_swings.sum())})
    sensitivity_rows.append(row)
threshold_sensitivity = pd.DataFrame(sensitivity_rows)
display(threshold_sensitivity.round(3))
threshold_sensitivity.to_csv(RESULTS / 'minimum_count_sensitivity.csv',index=False)

eligible_keys = set(zip(sessions.hitter,sessions.session))
groups = [(key,g.attack_angle_deg.to_numpy()) for key,g in trials.groupby(['hitter','session'])
          if key in eligible_keys]
rng = np.random.default_rng(SEED+400)
draw_records = []
for repetition in range(N_BALANCED):
    trial_sds = pd.DataFrame([{'hitter':key[0], 'sd':rng.choice(x,5,replace=False).std(ddof=1)}
                             for key,x in groups])
    per_hitter = trial_sds.groupby('hitter').sd.mean()
    t = hitters[['hitter','highest_playing_level']].copy()
    t['balanced_sd'] = t.hitter.map(per_hitter)
    a = t.loc[t.highest_playing_level.eq('milb'),'balanced_sd']
    b = t.loc[t.highest_playing_level.eq('college'),'balanced_sd']
    draw_records.append({'repetition':repetition,
                         'difference_deg':a.mean()-b.mean() if len(a) and len(b) else np.nan})
balanced_draws = pd.DataFrame(draw_records)
valid_differences = balanced_draws.difference_deg.dropna()
balanced_summary = pd.DataFrame([{
    'swings_per_session':5,'repetitions':N_BALANCED,
    'median_difference_deg':valid_differences.median(),
    'subsampling_low':valid_differences.quantile(.025),
    'subsampling_high':valid_differences.quantile(.975)}])
display(balanced_summary.round(3))
balanced_draws.to_csv(RESULTS / 'equal_five_trial_draws.csv',index=False)
balanced_summary.to_csv(RESULTS / 'equal_five_trial_summary.csv',index=False)


In [ ]:
contrast_cohort = hitters.loc[hitters.highest_playing_level.isin(['milb','college'])].copy()
full_sd = float(contrasts.loc[contrasts.metric.eq('angle_sd_deg'),'difference_deg'].iloc[0])
full_mad = float(contrasts.loc[contrasts.metric.eq('angle_mad_deg'),'difference_deg'].iloc[0])
loo_group_rows = []
for i,(_,omitted) in enumerate(contrast_cohort.sort_values('hitter').iterrows()):
    reduced = contrast_cohort.loc[contrast_cohort.hitter.ne(omitted.hitter)].copy()

    sd = group_contrast(reduced,'angle_sd_deg',SEED+600)
    a = reduced.loc[reduced.highest_playing_level.eq('milb'),'angle_mad_deg']
    b = reduced.loc[reduced.highest_playing_level.eq('college'),'angle_mad_deg']
    reduced_mad = a.mean()-b.mean() if len(a) and len(b) else np.nan
    valid_ci = np.isfinite(sd['ci_low']) and np.isfinite(sd['ci_high'])
    loo_group_rows.append({
        'omitted_hitter':omitted.hitter, 'omitted_level':omitted.highest_playing_level,
        'omitted_sd_deg':omitted.angle_sd_deg,'omitted_n_swings':int(omitted.n_swings),
        'n_milb_remaining':sd['n_milb'],'n_college_remaining':sd['n_college'],
        'difference_deg':sd['difference_deg'],'change_from_full_deg':sd['difference_deg']-full_sd,
        'ci_low':sd['ci_low'],'ci_high':sd['ci_high'],
        'interval_includes_zero':(sd['ci_low']<=0<=sd['ci_high']) if valid_ci else None,
        'mad_difference_deg':reduced_mad,'mad_change_from_full_deg':reduced_mad-full_mad})
loo_groups = pd.DataFrame(loo_group_rows)
loo_group_summary_rows = []
for level,g in loo_groups.groupby('omitted_level',sort=True):
    valid = g.dropna(subset=['difference_deg'])
    if valid.empty: continue
    top = valid.loc[valid.change_from_full_deg.abs().idxmax()]
    ci_valid = g.dropna(subset=['ci_low','ci_high'])
    loo_group_summary_rows.append({
        'omitted_level':level,'n_omissions':len(g), 'full_difference_deg':full_sd,
        'omission_difference_min':valid.difference_deg.min(),
        'omission_difference_max':valid.difference_deg.max(),
        'max_absolute_change_deg':valid.change_from_full_deg.abs().max(),
        'largest_change_hitter':top.omitted_hitter,
        'n_sign_reversals':int((np.sign(valid.difference_deg)!=np.sign(full_sd)).sum())
                          if full_sd != 0 and np.isfinite(full_sd) else np.nan,
        'n_valid_omission_intervals':len(ci_valid),
        'n_intervals_including_zero':int(((ci_valid.ci_low<=0)&(ci_valid.ci_high>=0)).sum()),
        'mad_difference_min':g.mad_difference_deg.min(),
        'mad_difference_max':g.mad_difference_deg.max()})
loo_group_summary = pd.DataFrame(loo_group_summary_rows)
loo_groups.to_csv(RESULTS/'leave_one_hitter_out_group_details.csv',index=False)
loo_group_summary.to_csv(RESULTS/'leave_one_hitter_out_group_summary.csv',index=False)
display(loo_group_summary.round(3))
display(loo_groups.loc[loo_groups.omitted_level.eq('milb')].round(3))

loo_association_rows = []
for label,outcome,ycol,ylabel in specs:
    ht = outcome_tables[label]
    if ht.empty: continue
    ht = ht.dropna(subset=['angle_sd_deg',ycol])
    full_rho = float(associations.loc[associations.relationship.eq(label),'rho'].iloc[0])
    for _,omitted in ht.sort_values('hitter').iterrows():
        reduced = ht.loc[ht.hitter.ne(omitted.hitter)]
        rho = spearman(reduced.angle_sd_deg,reduced[ycol])[0] if len(reduced)>=10 else np.nan
        loo_association_rows.append({
            'relationship':label,'omitted_hitter':omitted.hitter,
            'omitted_level':omitted.highest_playing_level,
            'n_remaining':len(reduced),'full_rho':full_rho,'omission_rho':rho,
            'change_from_full_rho':rho-full_rho})
loo_associations = pd.DataFrame(loo_association_rows,columns=[
    'relationship','omitted_hitter','omitted_level','n_remaining','full_rho',
    'omission_rho','change_from_full_rho'])
loo_association_summary_rows = []
for label,g in loo_associations.groupby('relationship',sort=True):
    valid = g.dropna(subset=['omission_rho','full_rho'])
    if valid.empty: continue
    top = valid.loc[valid.change_from_full_rho.abs().idxmax()]
    full_rho = valid.full_rho.iloc[0]
    loo_association_summary_rows.append({
        'relationship':label,'n_valid_omissions':len(valid),'full_rho':full_rho,
        'omission_rho_min':valid.omission_rho.min(),'omission_rho_max':valid.omission_rho.max(),
        'max_absolute_rho_change':valid.change_from_full_rho.abs().max(),
        'largest_change_hitter':top.omitted_hitter,'largest_change_level':top.omitted_level,
        'n_sign_reversals':int((np.sign(valid.omission_rho)!=np.sign(full_rho)).sum())
                          if full_rho != 0 else np.nan})
loo_association_summary = pd.DataFrame(loo_association_summary_rows)
loo_associations.to_csv(RESULTS/'leave_one_hitter_out_association_details.csv',index=False)
loo_association_summary.to_csv(RESULTS/'leave_one_hitter_out_association_summary.csv',index=False)
display(loo_association_summary.round(3))
print('Changes measure sensitivity to omissions; hitters are retained in the primary analysis.')


In [ ]:
loo_plot_levels = [l for l in LEVELS if l in set(hitters.highest_playing_level)]
loo_plot_levels += sorted(set(hitters.highest_playing_level)-set(loo_plot_levels))
fig,axes = plt.subplots(1,2,figsize=(13,5.5),gridspec_kw={'width_ratios':[1.4,1]})
ax = axes[0]
g = loo_groups.loc[loo_groups.omitted_level.eq('milb')].sort_values('difference_deg')
for i,(_,r) in enumerate(g.iterrows()):
    if np.isfinite(r.ci_low):
        ax.plot([r.ci_low,r.ci_high],[i,i],color=COLORS['milb'],lw=2)
    ax.scatter(r.difference_deg,i,color=COLORS['milb'],zorder=3,s=45)
ax.axvline(0,color='gray',lw=1)
ax.axvline(full_sd,color='black',ls='--',lw=1.4,label=f'Full estimate: {full_sd:+.2f}°')
ax.set_yticks(range(len(g)),[f'Omit {r.omitted_hitter}' for _,r in g.iterrows()])
ax.set(xlabel='MiLB minus college mean SD (degrees)',
       title='Omit each MiLB hitter: estimates and 95% intervals')
ax.legend(frameon=False,fontsize=9)
ax = axes[1]
rng = np.random.default_rng(SEED+601)
for i,level in enumerate(['milb','college']):
    g = loo_groups.loc[loo_groups.omitted_level.eq(level)]
    ax.scatter(i+rng.uniform(-.12,.12,len(g)),g.difference_deg,
               color=COLORS[level],alpha=.7,s=28)
ax.axhline(full_sd,color='black',ls='--',lw=1.4)
ax.axhline(0,color='gray',lw=1)
ax.set_xticks([0,1],['Omit MiLB hitter','Omit college hitter'])
ax.set(ylabel='MiLB minus college mean SD (degrees)',title='All individual omission estimates')
fig.tight_layout(); fig.savefig(FIGURES/'05_leave_one_hitter_out_groups.png'); plt.show()

fig,axes = plt.subplots(1,3,figsize=(14,4.5),sharey=True)
for ax,(label,outcome,ycol,ylabel) in zip(axes,specs):
    g = loo_associations.loc[loo_associations.relationship.eq(label)].dropna(subset=['omission_rho'])
    if g.empty:
        ax.text(.5,.5,'Insufficient data',ha='center',transform=ax.transAxes)
        continue
    g = g.sort_values('omission_rho')
    for level in loo_plot_levels:
        selected = g.omitted_level.eq(level).to_numpy()
        ax.scatter(np.arange(1,len(g)+1)[selected],g.loc[selected,'omission_rho'],
                   color=COLORS.get(level,'gray'),s=23,label=level.replace('_',' '))
    ax.axhline(g.full_rho.iloc[0],color='black',ls='--',lw=1.2)
    ax.axhline(0,color='gray',lw=1)
    ax.set(xlabel='Omissions ordered by resulting rho',title=ylabel)
axes[0].set_ylabel('Spearman rho after omitting one hitter')
axes[-1].legend(frameon=False,fontsize=8,title='Omitted hitter level')
fig.suptitle('Association influence: dashed line is full-cohort rho',y=1.01)
fig.tight_layout(); fig.savefig(FIGURES/'06_leave_one_hitter_out_associations.png',bbox_inches='tight'); plt.show()


In [ ]:
fig,ax = plt.subplots(figsize=(9,5))
plot_levels = [l for l in LEVELS if l in set(hitters.highest_playing_level)]
plot_levels += sorted(set(hitters.highest_playing_level)-set(plot_levels))
rng = np.random.default_rng(SEED+500)
for i,level in enumerate(plot_levels):
    g = hitters.loc[hitters.highest_playing_level.eq(level)]
    c = COLORS.get(level,'gray')
    ax.scatter(i+rng.uniform(-.13,.13,len(g)),g.angle_sd_deg,color=c,alpha=.65,s=30)
    row = level_summary.loc[level_summary.level.eq(level)].iloc[0]
    ax.scatter(i,row.mean_hitter_sd_deg,c='black',marker='D',s=55,zorder=5)
    if np.isfinite(row.ci_low):
        ax.plot([i,i],[row.ci_low,row.ci_high],color='black',lw=2)
ax.set_xticks(range(len(plot_levels)),[f'{l.replace("_"," ")}\n(n={sum(hitters.highest_playing_level.eq(l))})'
                                      for l in plot_levels])
ax.set(ylabel='Within-session attack-angle SD (degrees)',
       title='Attack-angle repeatability by recorded playing level')
ax.text(.01,.98,'Dots: hitters · Diamonds: group means · Lines: hitter-bootstrap 95% intervals',
        transform=ax.transAxes,va='top',fontsize=9)
fig.tight_layout(); fig.savefig(FIGURES/'01_playing_level_repeatability.png'); plt.show()

ranked = hitters.sort_values('angle_sd_deg').reset_index(drop=True)
fig,ax = plt.subplots(figsize=(10,4.8))
for level in plot_levels:
    g = ranked.loc[ranked.highest_playing_level.eq(level)]
    ax.scatter(g.index+1,g.angle_sd_deg,label=level.replace('_',' '),
               color=COLORS.get(level,'gray'),s=25)
ax.set(xlabel='Hitter order by observed SD',ylabel='Attack-angle SD (degrees)',
       title='Observed dispersion across hitters (short-session estimates)')
ax.legend(frameon=False,ncol=2); fig.tight_layout()
fig.savefig(FIGURES/'02_hitter_variability.png'); plt.show()

fig,axes = plt.subplots(1,3,figsize=(14,4.5))
for ax,(label,outcome,ycol,ylabel) in zip(axes,specs):
    ht = outcome_tables[label]
    if ht.empty:
        ax.text(.5,.5,'Insufficient paired data',ha='center',transform=ax.transAxes)
        continue
    for level in plot_levels:
        t = ht.loc[ht.highest_playing_level.eq(level)]
        ax.scatter(t.angle_sd_deg,t[ycol],color=COLORS.get(level,'gray'),alpha=.7,s=28)
    r = associations.loc[associations.relationship.eq(label)].iloc[0]
    ax.set(xlabel='Attack-angle SD (degrees)',ylabel=ylabel,
           title=f'rho={r.rho:.2f} | n={int(r.n_hitters)}')
fig.suptitle('Exploratory hitter-level associations; paired-trial cohorts',y=1.01)
fig.tight_layout(); fig.savefig(FIGURES/'03_outcome_associations.png',bbox_inches='tight'); plt.show()

ordered = sessions.sort_values('angle_sd_deg').reset_index(drop=True)
indices = np.unique(np.round(np.quantile(np.arange(len(ordered)),[.25,.5,.75])).astype(int))
fig,axes = plt.subplots(1,len(indices),figsize=(5*len(indices),4.4),squeeze=False,sharey=True)
for ax,idx in zip(axes[0],indices):
    s = ordered.iloc[idx]
    g = trials.loc[trials.hitter.eq(s.hitter)&trials.session.eq(s.session)].copy()

    g['swing_order'] = pd.to_numeric(g.session_swing.str.rsplit('_',n=1).str[-1],errors='coerce')
    g = g.sort_values(['swing_order','session_swing'])
    ax.plot(np.arange(1,len(g)+1),g.attack_angle_deg,'o-',color=COLORS.get(s.highest_playing_level,'gray'))
    ax.axhline(s.angle_mean_deg,color='gray',ls='--',lw=1)
    ax.set(xlabel='Recorded trial order',title=f'Hitter {s.hitter} · session {s.session}\nSD={s.angle_sd_deg:.2f}°; n={len(g)}')
axes[0,0].set_ylabel('Attack angle at contact (degrees)')
fig.suptitle('Representative session profiles; these are not bat-path curves',y=1.02)
fig.tight_layout(); fig.savefig(FIGURES/'04_example_trial_profiles.png',bbox_inches='tight'); plt.show()


In [ ]:
primary = contrasts.loc[contrasts.metric.eq('angle_sd_deg')].iloc[0]
if not np.isfinite(primary.ci_low):
    group_text = 'There were too few eligible hitters for the planned group interval.'
elif primary.ci_low <= 0 <= primary.ci_high:
    group_text = 'The interval includes zero; the direction of the playing-level difference is uncertain.'
else:
    direction = 'lower' if primary.difference_deg < 0 else 'higher'
    group_text = f'The sampled MiLB hitters had {direction} variability; this is exploratory and based on a small MiLB cohort.'
lines = [
    '# Attack-angle repeatability: run interpretation',
    f'- Retained {len(trials)} valid trials; primary analysis: {len(hitters)} hitters, {len(sessions)} sessions, {int(sessions.n_swings.sum())} swings.',
    f'- MiLB minus college mean hitter SD: {primary.difference_deg:+.2f} degrees; hitter-bootstrap 95% interval [{primary.ci_low:+.2f}, {primary.ci_high:+.2f}].',
    '- '+group_text,
    '- Group intervals are conditional on short-session variability estimates. Equal-five-trial ranges are subsampling ranges, not confidence intervals.',
]
for _,r in associations.iterrows():
    lines.append(f'- {r.relationship}: Spearman rho={r.rho:.3f}, 95% interval [{r.ci_low:.3f}, {r.ci_high:.3f}], n={int(r.n_hitters)}, raw p={r.p_value:.4f}, FDR q={r.fdr_q_value:.4f}.')
lines += [
    '', '## Interpretation limits',
    '- These measurements describe contact-angle repeatability in the recorded lab trials, not full swing-plane consistency, game performance, or long-term reliability.',
    '- Playing level is a contextual grouping; the data do not identify an elite MLB comparison cohort.',
    '- Pitch height/location and task demands can produce appropriate swing adjustments. This workflow does not adjust for those differences.',
    '- Fewer degrees of variability is not automatically better. No optimal angle, universal ranking, or mechanical prescription is established.',
    '- Outcome relationships are unadjusted, exploratory, and potentially confounded. Few trials make individual SDs uncertain.',
    '- Future stage: validated C3D bat reconstruction, official contact alignment, fixed pre-contact windows, separate 3D path-dispersion and local-plane-orientation metrics, and pitch-context adjustment.',
    '', '## Sources',
    '- OBP Dataset v1: https://github.com/drivelineresearch/openbiomechanics/releases/tag/dataset-v1',
    '- Hitting documentation: https://github.com/drivelineresearch/openbiomechanics/tree/main/baseball_hitting',
    '- Data license: https://github.com/drivelineresearch/openbiomechanics/blob/main/LICENSE-DATA.md',
]
lines += ['', '## Leave-one-hitter-out influence diagnostics']
for _,r in loo_group_summary.iterrows():
    lines.append(
        f'- Omitting each {r.omitted_level} hitter: SD differences range from '
        f'{r.omission_difference_min:+.2f} to {r.omission_difference_max:+.2f} degrees; '
        f'largest absolute change {r.max_absolute_change_deg:.2f} degrees '
        f'(hitter {r.largest_change_hitter}); {int(r.n_sign_reversals)} sign reversals. '
        f'{int(r.n_intervals_including_zero)} of {int(r.n_valid_omission_intervals)} '
        'available omission bootstrap intervals include zero.')
for _,r in loo_association_summary.iterrows():
    lines.append(
        f'- {r.relationship}: omission rho range [{r.omission_rho_min:.3f}, '
        f'{r.omission_rho_max:.3f}], largest absolute rho change '
        f'{r.max_absolute_rho_change:.3f} (hitter {r.largest_change_hitter}). '
        'Compare magnitude and original FDR results; sign changes near zero are not strong evidence.')
lines += ['- Omission ranges are influence ranges, not confidence intervals. '
          'Stable omission estimates do not establish causation or remove short-session uncertainty. '
          'These diagnostics do not justify deleting hitters or selecting a favorable subset.']
interpretation = '\n'.join(lines)+'\n'
(RESULTS/'interpretation.md').write_text(interpretation,encoding='utf-8')
versions = {p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','matplotlib']}
manifest = {'study':'Attack Angle Repeatability Stage 1','created_utc':datetime.now(timezone.utc).isoformat(),
            'dataset_target':'dataset-v1','python':platform.python_version(),'packages':versions,
            'seed':SEED,'minimum_swings':MIN_SWINGS,'bootstrap_repetitions':N_BOOT,
            'balanced_subsampling_repetitions':N_BALANCED,'inputs':input_records,
            'primary_metric':'sample within-session SD; mean of eligible session SDs per hitter',
            'primary_group_contrast':'MiLB minus college mean hitter SD',
            'association_tests':[s[0] for s in specs], 'github_modified':False, 'revision':'whole-hitter omission influence update',
            'influence_diagnostics':{'groups':'all MiLB and college hitters; SD intervals and MAD point estimates',
                                     'associations':'all hitters in each paired cohort; Spearman point estimates'}}
(RESULTS/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
display(Markdown(interpretation))
print('Saved results to:',RESULTS)
print('Saved figures to:',FIGURES)
